# Notebook 03 - Markov Chain Attribution (RQ2 Extension)

This notebook adds a Markov chain removal-effect benchmark for RQ2. 

The goal is to estimate how much the modeled conversion probability decreases when each channel is removed from the transition system.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd().resolve(), Path.cwd().resolve() / "notebooks", Path.cwd().resolve() / "analysis_python" / "notebooks", Path.cwd().resolve().parent / "notebooks"]:
    if (candidate / "notebook_header.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find notebook_header.py")

import numpy as np
import pandas as pd

from notebook_header import OUTPUT_DIR as BASE_OUTPUT_DIR, RANDOM_SEED, load_journeys

OUTPUT_DIR = BASE_OUTPUT_DIR / "rq2_markov"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.precision", 6)
np.random.seed(RANDOM_SEED)

## 1. Load Cleaned Journey Paths

In [2]:
df_jr = load_journeys()

assert len(df_jr) == 2_847
assert int(df_jr["Converted"].sum()) == 2_381
assert df_jr["Channel_Sequence"].notna().all()

channels = sorted(set(" -> ".join(df_jr["Channel_Sequence"]).split(" -> ")))
states = ["START"] + channels + ["CONVERSION", "NULL"]

print(f"Journeys: {df_jr.shape}")
print(f"Channels: {channels}")
print(f"Observed user-level conversion probability: {df_jr['Converted'].mean():.6f}")

Journeys: (2847, 9)
Channels: ['Direct Traffic', 'Display Ads', 'Email', 'Referral', 'Search Ads', 'Social Media']
Observed user-level conversion probability: 0.836319


## 2. Build Markov Transition Matrix

Each user journey is represented as `START -> channel sequence -> terminal state`. The terminal state is `CONVERSION` for converted users and `NULL` for non-converted users.

In [3]:
def journey_to_path(row):
    channel_sequence = str(row["Channel_Sequence"]).split(" -> ")
    terminal = "CONVERSION" if bool(row["Converted"]) else "NULL"
    return ["START"] + channel_sequence + [terminal]


def build_transition_matrix(paths, state_order):
    counts = pd.DataFrame(0.0, index=state_order, columns=state_order)
    for path in paths:
        for source, target in zip(path[:-1], path[1:]):
            counts.loc[source, target] += 1

    # Keep absorbing states valid for matrix algebra.
    counts.loc["CONVERSION", "CONVERSION"] = 1
    counts.loc["NULL", "NULL"] = 1

    probabilities = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return counts, probabilities


paths = [journey_to_path(row) for _, row in df_jr.iterrows()]
transition_counts, transition_matrix = build_transition_matrix(paths, states)

transition_matrix

,START,Direct Traffic,Display Ads,Email,Referral,Search Ads,Social Media,CONVERSION,NULL
START,0.0,0.171760,0.172813,0.158061,0.170706,0.161925,0.164735,0.000000,0.000000
Direct Traffic,0.0,0.126671,0.115630,0.117374,0.109239,0.119698,0.114468,0.246949,0.049971
Display Ads,0.0,0.115039,0.131815,0.124026,0.124026,0.115039,0.109646,0.240264,0.040144
Email,0.0,0.117896,0.114873,0.118501,0.116082,0.128174,0.117291,0.237606,0.049577
Referral,0.0,0.131157,0.127003,0.115134,0.112166,0.113947,0.130564,0.227893,0.042136
Search Ads,0.0,0.124922,0.104413,0.122436,0.126787,0.101927,0.121193,0.245494,0.052828
Social Media,0.0,0.123345,0.111913,0.125150,0.131769,0.109507,0.122744,0.230445,0.045126
CONVERSION,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
NULL,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000


## 3. Absorption Probability and Removal Effect

In [4]:
def conversion_absorption_probability(P):
    absorbing_states = ["CONVERSION", "NULL"]
    transient_states = [state for state in P.index if state not in absorbing_states]

    Q = P.loc[transient_states, transient_states].to_numpy(dtype=float)
    R = P.loc[transient_states, ["CONVERSION"]].to_numpy(dtype=float)
    N = np.linalg.solve(np.eye(len(transient_states)) - Q, R)
    return float(N[transient_states.index("START"), 0])


def remove_channel_from_transition_matrix(P, channel):
    """Remove a channel state and redistribute inbound probability through its outbound transitions."""
    P_removed = P.copy()
    outgoing = P_removed.loc[channel].copy()

    for state in list(P_removed.index):
        if state == channel:
            continue
        inbound = P_removed.loc[state, channel]
        if inbound != 0:
            P_removed.loc[state, :] = P_removed.loc[state, :] + inbound * outgoing
            P_removed.loc[state, channel] = 0

    P_removed = P_removed.drop(index=channel, columns=channel)
    P_removed = P_removed.div(P_removed.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    return P_removed


baseline_probability = conversion_absorption_probability(transition_matrix)

removal_rows = []
for channel in channels:
    removed_matrix = remove_channel_from_transition_matrix(transition_matrix, channel)
    probability_without_channel = conversion_absorption_probability(removed_matrix)
    removal_effect_abs = baseline_probability - probability_without_channel
    removal_rows.append({
        "channel": channel,
        "baseline_conversion_probability": baseline_probability,
        "conversion_probability_without_channel": probability_without_channel,
        "removal_effect_abs": removal_effect_abs,
        "removal_effect_pct": removal_effect_abs / baseline_probability * 100,
    })

markov_removal_effect = pd.DataFrame(removal_rows).sort_values("removal_effect_abs", ascending=False)
markov_removal_effect["positive_removal_effect"] = markov_removal_effect["removal_effect_abs"].clip(lower=0)

positive_total = markov_removal_effect["positive_removal_effect"].sum()
markov_removal_effect["markov_share_pct"] = np.where(
    positive_total > 0,
    markov_removal_effect["positive_removal_effect"] / positive_total * 100,
    0,
)

markov_removal_effect.round(6)

,channel,baseline_conversion_probability,conversion_probability_without_channel,removal_effect_abs,removal_effect_pct,positive_removal_effect,markov_share_pct
1,Display Ads,0.836319,0.835871,0.000448,0.053515,0.000448,76.056566
3,Referral,0.836319,0.836178,0.000141,0.016847,0.000141,23.943434
5,Social Media,0.836319,0.836320,-0.000001,-0.000079,0.000000,0.000000
0,Direct Traffic,0.836319,0.836429,-0.000110,-0.013203,0.000000,0.000000
2,Email,0.836319,0.836506,-0.000187,-0.022364,0.000000,0.000000
4,Search Ads,0.836319,0.836556,-0.000237,-0.028313,0.000000,0.000000


## 4. Compare Markov with Other RQ2 Benchmarks

In [5]:
model_comparison = pd.read_csv(OUTPUT_DIR.parent / "rq1_basic" / "01_attribution_share.csv", index_col=0)

logit_path = OUTPUT_DIR.parent / "rq2_logit" / "02_adjusted_channel_share.csv"
if logit_path.exists():
    logit_share = pd.read_csv(logit_path, index_col=0)["logit_adjusted_share_pct"]
    model_comparison["logit_adjusted_pct"] = logit_share

markov_share = markov_removal_effect.set_index("channel")["markov_share_pct"]
model_comparison["markov_positive_removal_pct"] = markov_share
model_comparison = model_comparison.fillna(0)

model_comparison

,first_touch_pct,last_touch_pct,linear_pct,logit_adjusted_pct,markov_positive_removal_pct
Direct Traffic,17.26,17.85,17.15,15.62,0.000000
Display Ads,17.98,16.84,17.10,21.99,76.056566
Referral,17.14,16.13,16.75,15.47,23.943434
Social Media,16.34,16.09,16.70,16.40,0.000000
Email,15.71,16.51,16.28,17.84,0.000000
Search Ads,15.58,16.59,16.02,12.68,0.000000


## 5. Save Outputs

In [6]:
transition_counts.to_csv(OUTPUT_DIR / "03_transition_counts.csv")
transition_matrix.to_csv(OUTPUT_DIR / "03_transition_matrix.csv")
markov_removal_effect.to_csv(OUTPUT_DIR / "03_removal_effect.csv", index=False)
markov_removal_effect[["channel", "markov_share_pct"]].to_csv(OUTPUT_DIR / "03_attribution_share.csv", index=False)
model_comparison.to_csv(OUTPUT_DIR / "03_model_comparison.csv")

print("Saved Markov outputs to outputs/rq2_markov/.")

Saved Markov outputs to outputs/rq2_markov/.


## 6. Markov/RQ2 conclusion

The Markov model serves as a supplementary check on the RQ2 findings rather than a replacement for the logistic benchmark. It successfully reproduces the observed user-level conversion probability of 83.63%, which suggests the fitted transition structure is internally consistent. The removal effects, however, remain very small in absolute terms. Removing Display Ads changes the modeled conversion probability by only 0.000448, or 0.0535%, while removing Referral changes it by 0.000141, or 0.0168%. This pattern is consistent with what was found in both RQ1 and the logistic analysis, where the channel mix was balanced and the channel signal was weak throughout.

After normalizing only positive removal effects, Display Ads receives 76.06% of Markov attribution and Referral receives the remaining 23.94%. Social Media, Direct Traffic, Email, and Search Ads all produce zero or negative removal effects in this transition system and therefore receive 0% under this normalization. This result needs some care in interpretation: a zero share does not imply those channels have no business value. It simply means that, within this particular modeled system, removing them does not reduce the estimated conversion probability. The large 76.06% share for Display Ads should also be read carefully because it is calculated from a very small pool of positive removal effects.

Taken together, the Markov result reinforces rather than extends the logistic benchmark findings. It is directionally consistent in pointing to Display Ads as the strongest channel, which also ranks first in the adjusted logistic benchmark at 21.99%. Still, the absolute removal effects are too small to support a strong claim that channels differ meaningfully in their contribution to conversion. That underlying uncertainty does not go away with the addition of the Markov model, and it is worth carrying that caveat forward into the discussion.
